# LeetCode 304 — Range Sum Query 2D - Immutable bằng Python

Link đề: https://leetcode.com/problems/range-sum-query-2d-immutable/ · Độ khó: **Medium** · Chủ đề: Design, Matrix, Prefix Sum

Skeleton LeetCode:

```python
class NumMatrix:
    def __init__(self, matrix: List[List[int]]):
        ...

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        ...

# Your NumMatrix object will be instantiated and called as such:
# obj = NumMatrix(matrix)
# param_1 = obj.sumRegion(row1,col1,row2,col2)
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Đây là **bài thiết kế lớp**: ta tự dựng **khung chạy test** kiểu LeetCode (danh sách tên hàm + danh sách đối số) và đối chiếu với một brute force độc lập (cộng trực tiếp các phần tử của hình chữ nhật bằng lát cắt).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế lớp `NumMatrix` nhận ma trận một lần, rồi trả lời **nhiều truy vấn** "tổng các phần tử trong hình chữ nhật có góc trên-trái `(row1, col1)` và góc dưới-phải `(row2, col2)`" (cả hai góc **thuộc** hình chữ nhật), mỗi truy vấn **`O(1)`**.
2. Hiểu thiết kế **"tiền xử lý một lần, truy vấn nhiều lần"** và cách đọc định dạng test bài design (`["NumMatrix","sumRegion",...]`).
3. Đi từ **cộng trực tiếp** `O(m·n)`/truy vấn → **tổng tiền tố theo từng hàng** `O(m)`/truy vấn → **tổng tiền tố 2D (bảng diện tích)** `O(1)`/truy vấn → **cây Fenwick 2D** (mở rộng sang bản **có cập nhật**, `O(log m · log n)`).
4. Nắm công thức **bao hàm – loại trừ**: `S(r1,c1,r2,c2) = P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]` với bảng `P` có **hàng/cột đệm bằng `0`**.
5. Tránh các lỗi kinh điển: **lệch chỉ số** (quên `+1`), **quên cộng lại** phần giao `P[r1][c1]`, không có **hàng/cột đệm** (chỉ số `-1` quay vòng trong Python), và dựng `P` sai (không trừ phần chồng lấn).

## 2. Đề bài & ví dụ

Cho ma trận 2 chiều `matrix`, hãy xử lý **nhiều truy vấn** dạng:

- Tính **tổng các phần tử** của `matrix` nằm trong hình chữ nhật xác định bởi **góc trên-trái** `(row1, col1)` và **góc dưới-phải** `(row2, col2)`.

Cài đặt lớp `NumMatrix`:

- `NumMatrix(int[][] matrix)` khởi tạo đối tượng với ma trận `matrix`.
- `int sumRegion(int row1, int col1, int row2, int col2)` trả về tổng các phần tử trong hình chữ nhật đó.

Bạn phải thiết kế thuật toán để `sumRegion` chạy trong **thời gian `O(1)`**.

**Ví dụ 1**

```
Input
["NumMatrix", "sumRegion", "sumRegion", "sumRegion"]
[[[[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]]], [2, 1, 4, 3], [1, 1, 2, 2], [1, 2, 2, 4]]
Output
[null, 8, 11, 12]
```

Giải thích: `sumRegion(2, 1, 4, 3)` trả `8`; `sumRegion(1, 1, 2, 2)` trả `11`; `sumRegion(1, 2, 2, 4)` trả `12`.

**Ràng buộc:** `m == matrix.length`; `n == matrix[i].length`; `1 <= m, n <= 200`; `-10^4 <= matrix[i][j] <= 10^4`; `0 <= row1 <= row2 < m`; `0 <= col1 <= col2 < n`; tối đa `10^4` lần gọi `sumRegion`.

In [1]:
from typing import List
import random
import time

M0 = [[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]]
examples_ops = ["NumMatrix", "sumRegion", "sumRegion", "sumRegion"]
examples_args = [[M0], [2, 1, 4, 3], [1, 1, 2, 2], [1, 2, 2, 4]]
examples_expected = [None, 8, 11, 12]


def run(cls, ops, args):
    """Chạy chuỗi thao tác kiểu LeetCode: phần tử đầu tạo đối tượng (kết quả None), các phần tử sau gọi phương thức."""
    obj = None
    out = []
    for op, arg in zip(ops, args):
        if obj is None:
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


print("ops     :", examples_ops)
print("kỳ vọng :", examples_expected)
for row in M0:
    print(" ", row)

ops     : ['NumMatrix', 'sumRegion', 'sumRegion', 'sumRegion']
kỳ vọng : [None, 8, 11, 12]
  [3, 0, 1, 4, 2]
  [5, 6, 3, 2, 1]
  [1, 2, 0, 1, 5]
  [4, 1, 0, 1, 7]
  [1, 0, 3, 0, 5]


## 3. Kiến thức nền

### 3.1. Lớp, `__init__` và "tiền xử lý một lần"

Với bài design, `__init__` chạy **một lần** khi tạo đối tượng và có thể làm việc nặng (tiền xử lý) để các phương thức sau (`sumRegion`) chạy **nhanh**. Nếu tổng số truy vấn là `Q` thì tổng thời gian là `thời gian tiền xử lý + Q × thời gian mỗi truy vấn`.
#### Code demo

In [2]:
class Demo:
    def __init__(self, values):
        self.prefix = [0]
        for v in values:
            self.prefix.append(self.prefix[-1] + v)          # tiền xử lý O(n)

    def range_sum(self, l, r):                               # truy vấn O(1)
        return self.prefix[r + 1] - self.prefix[l]


d = Demo([3, 0, 1, 4, 2])
print(d.range_sum(1, 3), d.range_sum(0, 4))                  # 5 10

5 10


### 3.2. Tổng tiền tố 1 chiều

`prefix[i]` = tổng `i` phần tử đầu. Khi đó tổng đoạn `[l, r]` là `prefix[r + 1] − prefix[l]`. `prefix[0] = 0` là phần tử **đệm** giúp đoạn bắt đầu từ `0` không cần xử lý riêng. Đây là nền của lời giải 2D.
#### Code demo

In [3]:
a = [3, 0, 1, 4, 2]
prefix = [0]
for v in a:
    prefix.append(prefix[-1] + v)
print(prefix)                                                # [0, 3, 3, 4, 8, 10]
print(prefix[4] - prefix[1], sum(a[1:4]))                    # 5 5: tổng a[1..3]

[0, 3, 3, 4, 8, 10]
5 5


### 3.3. Tổng tiền tố 2 chiều

Đặt `P[i][j]` = tổng mọi phần tử `matrix[r][c]` với `r < i` và `c < j` (hình chữ nhật từ góc `(0,0)` tới `(i-1, j-1)`). `P` có kích thước `(m+1) × (n+1)` với hàng `0` và cột `0` toàn `0` (đệm). Công thức dựng: `P[i+1][j+1] = matrix[i][j] + P[i][j+1] + P[i+1][j] − P[i][j]` (cộng phần **trên** và phần **trái**, **trừ** phần chéo bị đếm hai lần).
#### Code demo

In [4]:
def build_prefix(matrix):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = matrix[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P


P0 = build_prefix(M0)
for row in P0:
    print(row)
print("tổng cả ma trận:", P0[-1][-1], "=", sum(map(sum, M0)))

[0, 0, 0, 0, 0, 0]
[0, 3, 3, 4, 8, 10]
[0, 8, 14, 18, 24, 27]
[0, 9, 17, 21, 28, 36]
[0, 13, 22, 26, 34, 49]
[0, 14, 23, 30, 38, 58]
tổng cả ma trận: 58 = 58


### 3.4. Bao hàm – loại trừ cho hình chữ nhật

Tổng hình chữ nhật `(r1, c1) → (r2, c2)` bằng:

`P[r2+1][c2+1]` (cả khối từ gốc tới góc dưới-phải) `− P[r1][c2+1]` (bỏ dải phía trên) `− P[r2+1][c1]` (bỏ dải bên trái) `+ P[r1][c1]` (cộng lại góc giao bị trừ hai lần).
#### Code demo

In [5]:
def region_sum(P, r1, c1, r2, c2):
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]


print(region_sum(P0, 2, 1, 4, 3), region_sum(P0, 1, 1, 2, 2), region_sum(P0, 1, 2, 2, 4))     # 8 11 12

8 11 12


### 3.5. Vì sao cần hàng/cột đệm

Nếu không đệm, khi `r1 = 0` ta cần `P[r1 - 1]` = `P[-1]`: trong Python chỉ số `-1` **quay vòng** về phần tử **cuối**, cho kết quả sai **mà không báo lỗi**. Hàng/cột đệm `0` làm mọi truy vấn có công thức thống nhất.
#### Code demo

In [6]:
arr = [10, 20, 30]
print(arr[-1])                       # 30: KHÔNG phải "trước phần tử đầu tiên"

30


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Truy vấn `sumRegion(1, 1, 2, 2)` trên `M0`: hình chữ nhật hàng `1..2`, cột `1..2`:

```
5 [6 3] 2 1
1 [2 0] 1 5
```

Các phần tử `6, 3, 2, 0` ⇒ tổng `11` ✓. Theo bao hàm – loại trừ: `P[3][3]` (cả khối `3×3` góc trên-trái) `= 3+0+1+5+6+3+1+2+0 = 21`; trừ `P[1][3]` (hàng 0, cột 0..2 `= 3+0+1 = 4`) và `P[3][1]` (cột 0, hàng 0..2 `= 3+5+1 = 9`); cộng `P[1][1] = 3`: `21 − 4 − 9 + 3 = 11` ✓. **Cách nghĩ:** mọi hình chữ nhật là **hiệu** của bốn hình chữ nhật có góc ở gốc `(0, 0)`.

## 5. Cách 1 — Brute Force / baseline: cộng trực tiếp mọi phần tử

**Ý tưởng.** Lưu `matrix`; mỗi truy vấn duyệt `(row2 − row1 + 1) × (col2 − col1 + 1)` phần tử và cộng.
#### Code demo

In [7]:
class NumMatrixBrute:
    def __init__(self, matrix: List[List[int]]):
        self.matrix = matrix

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        total = 0
        for r in range(row1, row2 + 1):
            for c in range(col1, col2 + 1):
                total += self.matrix[r][c]
        return total


print(run(NumMatrixBrute, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 8, 11, 12] (kỳ vọng [None, 8, 11, 12] )


**Độ phức tạp (Cách 1).** Tiền xử lý `O(1)`; mỗi truy vấn **`O(m·n)`** worst (`200 × 200 = 4·10^4`); với `10^4` truy vấn là `4·10^8` — **quá chậm**. **Space `O(1)`** phụ.

**Nút thắt.** Ta cộng lại cùng các phần tử cho các truy vấn chồng lấn nhau. Tính trước **tổng tiền tố** để mỗi truy vấn chỉ còn vài phép tính.

## 6. Ý tưởng thuật toán / bước đột phá

**Tổng tiền tố 2D:** dựng một lần `P[(m+1) × (n+1)]` bằng công thức bao hàm – loại trừ ở 3.3; mỗi truy vấn chỉ cần **4 phép đọc và 3 phép cộng/trừ** theo 3.4 ⇒ **`O(1)`**. Tiền xử lý `O(m·n)` (một lần), bộ nhớ `O(m·n)`.

Bước trung gian là **tổng tiền tố theo từng hàng**: mỗi truy vấn cộng `m'` giá trị (`m'` số hàng của hình chữ nhật) ⇒ `O(m)`. Mở rộng cho **bản có cập nhật** là **cây Fenwick 2D**.

## 7. Các cách tiếp theo

### Cách 2 — Tổng tiền tố theo từng hàng

**Giải thích.** `row_prefix[r][j]` = tổng `matrix[r][0..j-1]`. Mỗi truy vấn: với mỗi hàng `r` từ `row1` tới `row2` cộng `row_prefix[r][col2 + 1] − row_prefix[r][col1]`.
#### Code demo

In [8]:
class NumMatrixRows:
    def __init__(self, matrix: List[List[int]]):
        self.row_prefix = []
        for row in matrix:
            prefix = [0]
            for v in row:
                prefix.append(prefix[-1] + v)
            self.row_prefix.append(prefix)

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        return sum(self.row_prefix[r][col2 + 1] - self.row_prefix[r][col1] for r in range(row1, row2 + 1))


print(run(NumMatrixRows, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 8, 11, 12] (kỳ vọng [None, 8, 11, 12] )


**Độ phức tạp Cách 2:** tiền xử lý `O(m·n)`; mỗi truy vấn **`O(m)`** (chỉ lặp theo hàng); **Space `O(m·n)`**. Với `10^4` truy vấn: `~2·10^6` — chạy được nhưng chưa đạt `O(1)`.

### Cách 3 — Tổng tiền tố 2D (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [9]:
class NumMatrix2D:
    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.P = [[0] * (n + 1) for _ in range(m + 1)]                  # hàng/cột đệm 0
        for i in range(m):
            for j in range(n):
                self.P[i + 1][j + 1] = matrix[i][j] + self.P[i][j + 1] + self.P[i + 1][j] - self.P[i][j]

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        P = self.P
        return P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1] + P[row1][col1]


print(run(NumMatrix2D, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 8, 11, 12] (kỳ vọng [None, 8, 11, 12] )


In [10]:
# Code nộp LeetCode cho Cách 3
class NumMatrix:
    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.s = [[0] * (n + 1) for _ in range(m + 1)]
        for i, row in enumerate(matrix):
            for j, v in enumerate(row):
                self.s[i + 1][j + 1] = self.s[i][j + 1] + self.s[i + 1][j] - self.s[i][j] + v

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        return (
            self.s[row2 + 1][col2 + 1]
            - self.s[row2 + 1][col1]
            - self.s[row1][col2 + 1]
            + self.s[row1][col1]
        )


print(run(NumMatrix, examples_ops, examples_args))

[None, 8, 11, 12]


**Độ phức tạp Cách 3:** tiền xử lý **`O(m·n)`**; mỗi truy vấn **`O(1)`**; **Space `O(m·n)`** (bảng `(m+1)(n+1)`).

### Cách 4 — Cây Fenwick 2D (mở rộng: hỗ trợ cập nhật)

**Giải thích.** **Cây Fenwick (BIT)** 2 chiều lưu tổng theo các khối "lowbit" và hỗ trợ **cập nhật một ô** và **tổng tiền tố** đều `O(log m · log n)`. Với bản **bất biến** (bài này) nó chậm hơn Cách 3 nhưng là nền để giải **Range Sum Query 2D - Mutable** (LeetCode 308). Ta cài đặt với phương thức `update(row, col, val)` thêm.
#### Code demo

In [11]:
class NumMatrixFenwick:
    def __init__(self, matrix: List[List[int]]):
        self.m, self.n = len(matrix), len(matrix[0])
        self.matrix = [row[:] for row in matrix]
        self.tree = [[0] * (self.n + 1) for _ in range(self.m + 1)]
        for r in range(self.m):
            for c in range(self.n):
                self._add(r + 1, c + 1, matrix[r][c])

    def _add(self, i, j, delta):
        while i <= self.m:
            jj = j
            while jj <= self.n:
                self.tree[i][jj] += delta
                jj += jj & -jj
            i += i & -i

    def _prefix(self, i, j):
        total = 0
        while i > 0:
            jj = j
            while jj > 0:
                total += self.tree[i][jj]
                jj -= jj & -jj
            i -= i & -i
        return total

    def update(self, row: int, col: int, val: int) -> None:
        self._add(row + 1, col + 1, val - self.matrix[row][col])
        self.matrix[row][col] = val

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        return (self._prefix(row2 + 1, col2 + 1) - self._prefix(row1, col2 + 1)
                - self._prefix(row2 + 1, col1) + self._prefix(row1, col1))


print(run(NumMatrixFenwick, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 8, 11, 12] (kỳ vọng [None, 8, 11, 12] )


**Độ phức tạp Cách 4:** dựng `O(m·n·log m·log n)` (bản đơn giản), mỗi `sumRegion` và `update` **`O(log m · log n)`** (`~ 8 × 8 = 64` thao tác với `200 × 200`); **Space `O(m·n)`**.

> Cách 3 là bản nộp chuẩn (`O(1)`). Cách 2 là bước đệm dễ hiểu. Cách 4 mở rộng được khi ma trận thay đổi.

## 8. TRACE từng bước

Ta in bảng tổng tiền tố `P` của `M0` và chi tiết một truy vấn: bốn số hạng và kết quả.
#### Code demo

In [12]:
def trace_query(P, r1, c1, r2, c2):
    a, b, c, d = P[r2 + 1][c2 + 1], P[r1][c2 + 1], P[r2 + 1][c1], P[r1][c1]
    print(f"sumRegion({r1},{c1},{r2},{c2}):")
    print(f"  + P[{r2 + 1}][{c2 + 1}] = {a}   (khối từ gốc tới ({r2},{c2}))")
    print(f"  - P[{r1}][{c2 + 1}] = {b}   (bỏ dải phía trên hàng {r1})")
    print(f"  - P[{r2 + 1}][{c1}] = {c}   (bỏ dải bên trái cột {c1})")
    print(f"  + P[{r1}][{c1}] = {d}   (cộng lại góc bị trừ hai lần)")
    print(f"  = {a} - {b} - {c} + {d} = {a - b - c + d}")
    return a - b - c + d


print("bảng P của M0 (hàng/cột 0 là đệm):")
for row in P0:
    print("  ", [f"{v:>3}" for v in row])
for q in examples_args[1:]:
    trace_query(P0, *q)

bảng P của M0 (hàng/cột 0 là đệm):
   ['  0', '  0', '  0', '  0', '  0', '  0']
   ['  0', '  3', '  3', '  4', '  8', ' 10']
   ['  0', '  8', ' 14', ' 18', ' 24', ' 27']
   ['  0', '  9', ' 17', ' 21', ' 28', ' 36']
   ['  0', ' 13', ' 22', ' 26', ' 34', ' 49']
   ['  0', ' 14', ' 23', ' 30', ' 38', ' 58']
sumRegion(2,1,4,3):
  + P[5][4] = 38   (khối từ gốc tới (4,3))
  - P[2][4] = 24   (bỏ dải phía trên hàng 2)
  - P[5][1] = 14   (bỏ dải bên trái cột 1)
  + P[2][1] = 8   (cộng lại góc bị trừ hai lần)
  = 38 - 24 - 14 + 8 = 8
sumRegion(1,1,2,2):
  + P[3][3] = 21   (khối từ gốc tới (2,2))
  - P[1][3] = 4   (bỏ dải phía trên hàng 1)
  - P[3][1] = 9   (bỏ dải bên trái cột 1)
  + P[1][1] = 3   (cộng lại góc bị trừ hai lần)
  = 21 - 4 - 9 + 3 = 11
sumRegion(1,2,2,4):
  + P[3][5] = 36   (khối từ gốc tới (2,4))
  - P[1][5] = 10   (bỏ dải phía trên hàng 1)
  - P[3][2] = 17   (bỏ dải bên trái cột 2)
  + P[1][2] = 3   (cộng lại góc bị trừ hai lần)
  = 36 - 10 - 17 + 3 = 12


Bảng tóm tắt (truy vấn `sumRegion(2, 1, 4, 3)`):

| số hạng | giá trị |
|---|---|
| `P[5][4]` | `38` (tổng khối từ gốc tới `(4, 3)`) |
| `− P[2][4]` | `− 24` (dải phía trên hàng `2`) |
| `− P[5][1]` | `− 14` (dải bên trái cột `1`) |
| `+ P[2][1]` | `+ 8` (góc giao, bị trừ hai lần) |
| **kết quả** | **`38 − 24 − 14 + 8 = 8`** |

Đúng với đáp án `8` của đề.

## 9. Code nộp LeetCode

Bản nộp dùng tổng tiền tố 2D (Cách 3). Đúng tên lớp `NumMatrix`, `__init__(self, matrix)`, `sumRegion(self, row1, col1, row2, col2)`.

In [13]:
class NumMatrix:
    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.s = [[0] * (n + 1) for _ in range(m + 1)]                 # s[i][j]: tổng khối hàng < i, cột < j (có đệm)
        for i, row in enumerate(matrix):
            for j, v in enumerate(row):
                self.s[i + 1][j + 1] = self.s[i][j + 1] + self.s[i + 1][j] - self.s[i][j] + v

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        return (
            self.s[row2 + 1][col2 + 1]
            - self.s[row2 + 1][col1]
            - self.s[row1][col2 + 1]
            + self.s[row1][col1]
        )


assert run(NumMatrix, examples_ops, examples_args) == examples_expected
print("OK: ví dụ của đề")

OK: ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Công thức dựng `P`:** `P[i+1][j+1]` là tổng khối `[0..i] × [0..j]` = (khối `[0..i-1] × [0..j]`) + (khối `[0..i] × [0..j-1]`) − (khối `[0..i-1] × [0..j-1]`, bị cộng hai lần) + `matrix[i][j]` (ô góc). Khối `[0..i-1] × [0..j]` là `P[i][j+1]`; khối `[0..i] × [0..j-1]` là `P[i+1][j]`; khối giao là `P[i][j]`.
- **Công thức truy vấn:** hình chữ nhật `[r1..r2] × [c1..c2]` = khối `[0..r2] × [0..c2]` − khối `[0..r1-1] × [0..c2]` − khối `[0..r2] × [0..c1-1]` + khối `[0..r1-1] × [0..c1-1]` (bao hàm – loại trừ: phần góc `[0..r1-1] × [0..c1-1]` bị trừ hai lần nên cộng lại một lần). Dịch chỉ số `+1` do đệm: `P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]`.
- **Đệm:** `P[0][*] = P[*][0] = 0` là tổng khối rỗng, nên công thức đúng cả khi `r1 = 0` hoặc `c1 = 0`.
- **Mỗi truy vấn `O(1)`:** đúng 4 lần đọc bảng.
- **Cách 4 (Fenwick):** `_prefix(i, j)` trả đúng tổng khối `[1..i] × [1..j]` (tính theo lowbit hai chiều) nên công thức bao hàm – loại trừ như trên; `update` cộng chênh lệch `val − cũ`.

## 11. Phân tích Time / Space Complexity

Gọi `m, n` kích thước ma trận (tối đa `200`), `Q` số truy vấn (tối đa `10^4`).

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Cộng trực tiếp | `O(1)` | `O(m·n)` | `O(1)` phụ |
| 2. Tổng tiền tố từng hàng | `O(m·n)` | `O(m)` | `O(m·n)` |
| 3. Tổng tiền tố 2D | `O(m·n)` | **`O(1)`** | `O(m·n)` |
| 4. Fenwick 2D | `O(m·n·log m·log n)` | `O(log m · log n)` | `O(m·n)` |

Tổng thời gian toàn bộ: Cách 1 `O(Q·m·n) ≈ 4·10^8`; Cách 3 `O(m·n + Q) ≈ 5·10^4`. Không có khác biệt average/worst: thời gian truy vấn Cách 3 luôn đúng 4 lần đọc.

## 12. Bộ test

`oracle` là brute force độc lập: cộng bằng **lát cắt** `sum(sum(row[c1:c2+1]) for row in matrix[r1:r2+1])`, trực tiếp theo định nghĩa. Ta sinh ma trận và truy vấn ngẫu nhiên và đối chiếu mọi cách **từng truy vấn**.
#### Code demo

In [14]:
def oracle(matrix, r1, c1, r2, c2):
    return sum(sum(row[c1:c2 + 1]) for row in matrix[r1:r2 + 1])


impls = {
    "brute": NumMatrixBrute,
    "rows": NumMatrixRows,
    "prefix2d": NumMatrix2D,
    "fenwick": NumMatrixFenwick,
    "submit": NumMatrix,
}

for key, cls in impls.items():
    assert run(cls, examples_ops, examples_args) == examples_expected, key
assert [oracle(M0, *q) for q in examples_args[1:]] == examples_expected[1:]
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([[5]], [(0, 0, 0, 0)], "ma trận 1x1"),
    ([[1, 2, 3]], [(0, 0, 0, 2), (0, 1, 0, 1), (0, 2, 0, 2)], "một hàng"),
    ([[1], [2], [3]], [(0, 0, 2, 0), (1, 0, 2, 0), (1, 0, 1, 0)], "một cột"),
    ([[1, 2], [3, 4]], [(0, 0, 1, 1), (0, 0, 0, 0), (1, 1, 1, 1), (0, 1, 1, 1), (1, 0, 1, 1)], "2x2: nhiều dạng góc"),
    ([[-10000] * 3 for _ in range(3)], [(0, 0, 2, 2), (1, 1, 1, 1)], "giá trị biên âm"),
    ([[10000] * 3 for _ in range(3)], [(0, 0, 2, 2)], "giá trị biên dương"),
    ([[0] * 4 for _ in range(4)], [(0, 0, 3, 3), (2, 2, 3, 3)], "toàn 0"),
    ([[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]],
     [(0, 0, 4, 4), (0, 0, 0, 4), (4, 0, 4, 4), (0, 4, 4, 4), (2, 2, 2, 2)], "ma trận ví dụ: cả bảng, hàng, cột, ô"),
]
for matrix, queries, label in edge_cases:
    ops = ["NumMatrix"] + ["sumRegion"] * len(queries)
    args = [[matrix]] + [list(q) for q in queries]
    expected = [None] + [oracle(matrix, *q) for q in queries]
    for key, cls in impls.items():
        assert run(cls, ops, args) == expected, (key, label)
    print("PASS", label, "->", expected[1:])

PASS ma trận 1x1 -> [5]
PASS một hàng -> [6, 2, 3]
PASS một cột -> [6, 5, 2]
PASS 2x2: nhiều dạng góc -> [10, 1, 4, 6, 7]
PASS giá trị biên âm -> [-90000, -10000]
PASS giá trị biên dương -> [90000]
PASS toàn 0 -> [0, 0]
PASS ma trận ví dụ: cả bảng, hàng, cột, ô -> [58, 10, 9, 20, 0]


### 12.2. Tính chất và phản ví dụ

- Tổng cả ma trận = `P[m][n]`; tổng một ô `(r, c)` là `matrix[r][c]`.
- **Cộng được:** hai hình chữ nhật kề nhau cộng lại bằng hình chữ nhật lớn.
- **Bất biến khi chuyển vị** (đổi hàng ↔ cột).
- **Phản ví dụ** cho lỗi không đệm / sai dấu: bản sai dùng `P[r1 - 1]`, `P[.][c1 - 1]` quay vòng khi `r1 = 0` hoặc `c1 = 0`.
#### Code demo

In [16]:
rng = random.Random(304)


def random_matrix(m, n, rng, lo=-10_000, hi=10_000):
    return [[rng.randint(lo, hi) for _ in range(n)] for _ in range(m)]


for _ in range(200):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    matrix = random_matrix(m, n, rng)
    nm = NumMatrix(matrix)
    assert nm.sumRegion(0, 0, m - 1, n - 1) == sum(map(sum, matrix))
    r, c = rng.randrange(m), rng.randrange(n)
    assert nm.sumRegion(r, c, r, c) == matrix[r][c]
    # cộng được: tách theo một hàng cắt
    if m >= 2:
        cut = rng.randint(1, m - 1)
        assert nm.sumRegion(0, 0, cut - 1, n - 1) + nm.sumRegion(cut, 0, m - 1, n - 1) == nm.sumRegion(0, 0, m - 1, n - 1)
    # chuyển vị
    tr = [list(col) for col in zip(*matrix)]
    r1, r2 = sorted((rng.randrange(m), rng.randrange(m)))
    c1, c2 = sorted((rng.randrange(n), rng.randrange(n)))
    assert NumMatrix(tr).sumRegion(c1, r1, c2, r2) == nm.sumRegion(r1, c1, r2, c2)
print("PASS tính chất (cả bảng, một ô, cộng được, chuyển vị) trên 200 ma trận ngẫu nhiên")


class NumMatrixSai:
    def __init__(self, matrix):
        m, n = len(matrix), len(matrix[0])
        self.P = [[0] * n for _ in range(m)]                           # SAI: không có hàng/cột đệm
        for i in range(m):
            for j in range(n):
                self.P[i][j] = matrix[i][j] + (self.P[i - 1][j] if i else 0) + (self.P[i][j - 1] if j else 0) - (self.P[i - 1][j - 1] if i and j else 0)

    def sumRegion(self, row1, col1, row2, col2):
        P = self.P
        return P[row2][col2] - P[row1 - 1][col2] - P[row2][col1 - 1] + P[row1 - 1][col1 - 1]      # SAI: chỉ số -1 quay vòng


q_ops = ["NumMatrix", "sumRegion", "sumRegion", "sumRegion"]
q_args = [[M0], [0, 0, 1, 1], [0, 2, 2, 4], [1, 0, 2, 3]]          # các truy vấn có row1 = 0 hoặc col1 = 0
print("đúng       :", run(NumMatrix, q_ops, q_args)[1:], "(oracle", [oracle(M0, *q) for q in q_args[1:]], ")")
print("không đệm  :", run(NumMatrixSai, q_ops, q_args)[1:])
assert run(NumMatrixSai, q_ops, q_args)[1:] != run(NumMatrix, q_ops, q_args)[1:]
print("(ví dụ của đề chỉ dùng row1, col1 >= 1 nên bản không đệm vô tình đúng:", run(NumMatrixSai, examples_ops, examples_args)[1:], ")")

PASS tính chất (cả bảng, một ô, cộng được, chuyển vị) trên 200 ma trận ngẫu nhiên
đúng       : [14, 19, 20] (oracle [14, 19, 20] )
không đệm  : [22, -16, -6]
(ví dụ của đề chỉ dùng row1, col1 >= 1 nên bản không đệm vô tình đúng: [8, 11, 12] )


### 12.3. Đối chiếu ngẫu nhiên (có seed), bản có cập nhật và dữ liệu lớn
#### Code demo

In [17]:
rng = random.Random(4304)
for trial in range(300):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    matrix = random_matrix(m, n, rng)
    q = rng.randint(1, 15)
    queries = []
    for _ in range(q):
        r1, r2 = sorted((rng.randrange(m), rng.randrange(m)))
        c1, c2 = sorted((rng.randrange(n), rng.randrange(n)))
        queries.append((r1, c1, r2, c2))
    ops = ["NumMatrix"] + ["sumRegion"] * q
    args = [[matrix]] + [list(x) for x in queries]
    expected = [None] + [oracle(matrix, *x) for x in queries]
    for key, cls in impls.items():
        assert run(cls, ops, args) == expected, (key, matrix, queries)
print("PASS 300 ma trận ngẫu nhiên x truy vấn ngẫu nhiên: mọi cách khớp oracle")

# Bản có cập nhật (Fenwick 2D) đối chiếu với brute force khi xen kẽ update và truy vấn
for trial in range(100):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    matrix = random_matrix(m, n, rng)
    fen = NumMatrixFenwick(matrix)
    shadow = [row[:] for row in matrix]
    for _ in range(40):
        if rng.random() < 0.4:
            r, c, v = rng.randrange(m), rng.randrange(n), rng.randint(-10_000, 10_000)
            fen.update(r, c, v)
            shadow[r][c] = v
        else:
            r1, r2 = sorted((rng.randrange(m), rng.randrange(m)))
            c1, c2 = sorted((rng.randrange(n), rng.randrange(n)))
            assert fen.sumRegion(r1, c1, r2, c2) == oracle(shadow, r1, c1, r2, c2)
print("PASS Fenwick 2D có cập nhật: khớp brute force trên 100 kịch bản xen kẽ update/truy vấn")

# Dữ liệu cỡ đề: 200 x 200, 10^4 truy vấn
m = n = 200
big = random_matrix(m, n, rng)
queries = []
for _ in range(10_000):
    r1, r2 = sorted((rng.randrange(m), rng.randrange(m)))
    c1, c2 = sorted((rng.randrange(n), rng.randrange(n)))
    queries.append((r1, c1, r2, c2))
ops = ["NumMatrix"] + ["sumRegion"] * len(queries)
args = [[big]] + [list(x) for x in queries]
ref = None
for key in ("prefix2d", "submit", "fenwick", "rows"):
    t0 = time.perf_counter()
    result = run(impls[key], ops, args)
    dt = (time.perf_counter() - t0) * 1000
    ref = result if ref is None else ref
    assert result == ref, key
    print(f"{key:<9} {dt:9.1f} ms (dựng + 10^4 truy vấn)")
sample = rng.sample(range(len(queries)), 200)
assert all(ref[1 + i] == oracle(big, *queries[i]) for i in sample)
t0 = time.perf_counter()
brute_small = run(NumMatrixBrute, ["NumMatrix"] + ["sumRegion"] * 300, [[big]] + [list(x) for x in queries[:300]])
print(f"brute     {(time.perf_counter() - t0) * 1000:9.1f} ms chỉ cho 300 truy vấn (10^4 truy vấn sẽ chậm gấp ~33 lần)")
assert brute_small == ref[:301]
print("PASS ma trận 200 x 200, 10^4 truy vấn")

PASS 300 ma trận ngẫu nhiên x truy vấn ngẫu nhiên: mọi cách khớp oracle
PASS Fenwick 2D có cập nhật: khớp brute force trên 100 kịch bản xen kẽ update/truy vấn
prefix2d        7.8 ms (dựng + 10^4 truy vấn)
submit          7.0 ms (dựng + 10^4 truy vấn)


fenwick       113.2 ms (dựng + 10^4 truy vấn)


rows           47.7 ms (dựng + 10^4 truy vấn)
brute          34.2 ms chỉ cho 300 truy vấn (10^4 truy vấn sẽ chậm gấp ~33 lần)
PASS ma trận 200 x 200, 10^4 truy vấn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không có hàng/cột đệm (chỉ số `-1` quay vòng)
Xem `NumMatrixSai` ở phần test: khi `row1 = 0` hay `col1 = 0`, `P[row1 - 1]` hay `P[.][col1 - 1]` đọc phần tử **cuối** của danh sách (không báo lỗi) và kết quả sai. **Code đúng:** `P` kích thước `(m+1) × (n+1)` với hàng/cột `0` bằng `0`.

### Lỗi 2: Quên cộng lại phần giao `P[row1][col1]`
#### Code sai

In [18]:
class NumMatrixQuenCong:
    def __init__(self, matrix):
        self.inner = NumMatrix2D(matrix)

    def sumRegion(self, row1, col1, row2, col2):
        P = self.inner.P
        return P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1]          # SAI: thiếu + P[row1][col1]


print("đúng:", run(NumMatrix, examples_ops, examples_args)[1:], "| sai:", run(NumMatrixQuenCong, examples_ops, examples_args)[1:])

đúng: [8, 11, 12] | sai: [0, 8, 9]


**Vì sao sai:** phần góc trên-trái `[0..r1-1] × [0..c1-1]` bị trừ **hai lần** (một lần trong "dải phía trên", một lần trong "dải bên trái"); phải **cộng lại** một lần.

### Lỗi 3: Dựng `P` sai (không trừ phần chồng lấn)
#### Code sai

In [19]:
class NumMatrixDungSai:
    def __init__(self, matrix):
        m, n = len(matrix), len(matrix[0])
        self.P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                self.P[i + 1][j + 1] = matrix[i][j] + self.P[i][j + 1] + self.P[i + 1][j]      # SAI: thiếu - P[i][j]

    def sumRegion(self, row1, col1, row2, col2):
        P = self.P
        return P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1] + P[row1][col1]


print("đúng:", run(NumMatrix, examples_ops, examples_args)[1:], "| sai:", run(NumMatrixDungSai, examples_ops, examples_args)[1:])

đúng: [8, 11, 12] | sai: [295, 42, 102]


**Vì sao sai:** khối "trên" và khối "trái" chồng lên nhau ở khối giao `P[i][j]`; cộng cả hai thì đếm giao **hai lần**, phải trừ đi một lần.

### Lỗi 4: Lệch chỉ số (quên `+1` khi dùng bảng đệm)
#### Code sai

In [20]:
class NumMatrixLechChiSo:
    def __init__(self, matrix):
        self.inner = NumMatrix2D(matrix)

    def sumRegion(self, row1, col1, row2, col2):
        P = self.inner.P
        return P[row2][col2] - P[row1][col2] - P[row2][col1] + P[row1][col1]            # SAI: thiếu +1 cho row2, col2


print("đúng:", run(NumMatrix, examples_ops, examples_args)[1:], "| sai:", run(NumMatrixLechChiSo, examples_ops, examples_args)[1:])

đúng: [8, 11, 12] | sai: [3, 6, 5]


**Vì sao sai:** với bảng đệm, khối kết thúc ở hàng `r2` có chỉ số `r2 + 1` (vì `P[i]` là tổng `i` hàng đầu). Quên `+1` làm bỏ sót hàng/cột cuối của hình chữ nhật.

### Lỗi 5: Tính lại tổng ở mỗi truy vấn (không tiền xử lý)
Cách 1 và Cách 2 đúng nhưng không đạt `O(1)`: yêu cầu của đề là `sumRegion` chạy `O(1)`; với `10^4` truy vấn trên `200 × 200` cách ngây thơ tốn hàng trăm triệu phép. **Cách sửa:** tiền xử lý một lần trong `__init__`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**nhiều truy vấn tổng trên đoạn / hình chữ nhật, dữ liệu không đổi**" → **tổng tiền tố** (1D: `P[r+1] − P[l]`; 2D: bao hàm – loại trừ).
- Luôn dùng **bảng đệm** (`+1` hàng/cột) để công thức không có trường hợp biên.
- Công thức 2D nhớ theo hình: **khối lớn − dải trên − dải trái + góc**.
- Dữ liệu **thay đổi** → **Fenwick / Segment tree**; không đổi → tổng tiền tố.
- Họ hàng: **Range Sum Query - Immutable** (303), **Range Sum Query 2D - Mutable** (308), **Matrix Block Sum** (1314), **Maximal Square** (221), **Number of Submatrices That Sum to Target** (1074).
- Bài design: tự dựng `run(cls, ops, args)` để test đúng định dạng LeetCode.

## 15. Biến thể của bài

1. **Range Sum Query 2D - Mutable** (LeetCode 308): thêm `update` — Cách 4.
2. **Matrix Block Sum** (1314): tổng các khối `(2k+1) × (2k+1)` quanh mỗi ô.
3. **Tổng tiền tố 3 chiều**: bao hàm – loại trừ 8 số hạng.
4. **Number of Submatrices That Sum to Target** (1074): kết hợp tổng tiền tố với bảng băm.
5. **Tổng lớn nhất của hình chữ nhật con** (Max Sum of Rectangle No Larger Than K — 363).
6. **Tổng tiền tố theo điều kiện** (đếm số ô thỏa điều kiện trong hình chữ nhật): đếm bằng tổng tiền tố của ma trận chỉ thị.
#### Code demo

In [21]:
def matrix_block_sum(mat, k):
    """LeetCode 1314: answer[i][j] = tổng các mat[r][c] với |r - i| <= k và |c - j| <= k."""
    m, n = len(mat), len(mat[0])
    nm = NumMatrix(mat)
    return [[nm.sumRegion(max(0, i - k), max(0, j - k), min(m - 1, i + k), min(n - 1, j + k)) for j in range(n)] for i in range(m)]


def count_positive_in_region(matrix):
    """Đếm số ô dương trong hình chữ nhật bằng tổng tiền tố của ma trận chỉ thị."""
    indicator = [[1 if v > 0 else 0 for v in row] for row in matrix]
    nm = NumMatrix(indicator)
    return lambda r1, c1, r2, c2: nm.sumRegion(r1, c1, r2, c2)


print(matrix_block_sum([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))
cnt = count_positive_in_region([[1, -2, 3], [-4, 5, -6]])
print(cnt(0, 0, 1, 2), cnt(0, 0, 0, 1))                # 3 1

[[12, 21, 16], [27, 45, 33], [24, 39, 28]]
3 1


## 16. So sánh các cách làm

| Cách | Tiền xử lý | Mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cộng trực tiếp | `O(1)` | `O(m·n)` | `O(1)` | ít truy vấn, oracle |
| 2. Tổng tiền tố từng hàng | `O(m·n)` | `O(m)` | `O(m·n)` | bước đệm dễ hiểu |
| 3. Tổng tiền tố 2D | `O(m·n)` | **`O(1)`** | `O(m·n)` | **cách nộp chuẩn** |
| 4. Fenwick 2D | `O(mn log m log n)` | `O(log m · log n)` | `O(m·n)` | khi có `update` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `build_prefix([[1, 2], [3, 4]])`; (b) `NumMatrix([[1, 2], [3, 4]]).sumRegion(0, 0, 1, 1)`; (c) `sumRegion(1, 0, 1, 1)`; (d) `sumRegion(0, 1, 1, 1)`.

**Bài 2 (tính bằng tay).** Với `matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]`, tính bảng `P` (có đệm) và `sumRegion(1, 1, 2, 2)` bằng công thức bao hàm – loại trừ.

**Bài 3 (điền TODO).** Hoàn thiện `NumMatrixTodo` theo Cách 3.
#### Code demo

In [22]:
class NumMatrixTodo:
    def __init__(self, matrix):
        # TODO 1: m, n = len(matrix), len(matrix[0]); P = (m + 1) x (n + 1) toàn 0
        # TODO 2: P[i + 1][j + 1] = matrix[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
        pass

    def sumRegion(self, row1, col1, row2, col2):
        # TODO 3: trả P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1] + P[row1][col1]
        pass


# Tự làm xong rồi mới mở test:
# assert run(NumMatrixTodo, examples_ops, examples_args) == examples_expected

**Bài 4 (tự viết lại).** Viết `NumMatrix1DRows` (Cách 2) **không nhìn lại**, và đo xem với `10^4` truy vấn nó chậm hơn Cách 3 khoảng bao nhiêu lần.

**Bài 5 (biến thể / mini challenge).** Viết `NumMatrix3D` cho khối 3 chiều `a × b × c` với `sumRegion(x1, y1, z1, x2, y2, z2)` `O(1)`: tổng tiền tố 3D và bao hàm – loại trừ **8** số hạng (dấu theo tính chẵn lẻ số chỉ số "bị trừ một"). Đối chiếu với cộng trực tiếp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: nhớ hàng/cột `0` là đệm. Bài 2: dựng `P` từng ô. Bài 5: tổng tiền tố 3D `P[i+1][j+1][k+1]` dùng bao hàm – loại trừ 7 số hạng.
- **Mức 2 (vừa):** Bài 1: `P = [[0,0,0],[0,1,3],[0,4,10]]`, `10`, `7`, `6`. Bài 2: `P` (có đệm) hàng 1: `[0,1,3,6]`; hàng 2: `[0,5,12,21]`; hàng 3: `[0,12,27,45]`; `sumRegion(1,1,2,2) = P[3][3] − P[1][3] − P[3][1] + P[1][1] = 45 − 6 − 12 + 1 = 28`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 5: `S = Σ (−1)^{(số chỉ số dùng cận dưới)} P[...]`: với mỗi trục chọn `hi + 1` (dấu `+`) hoặc `lo` (dấu `−`); tích dấu là dấu của số hạng. Bài 4: đo bằng `time.perf_counter`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
# Bài 1
assert build_prefix([[1, 2], [3, 4]]) == [[0, 0, 0], [0, 1, 3], [0, 4, 10]]
nm = NumMatrix([[1, 2], [3, 4]])
assert nm.sumRegion(0, 0, 1, 1) == 10 and nm.sumRegion(1, 0, 1, 1) == 7 and nm.sumRegion(0, 1, 1, 1) == 6

# Bài 2
m3 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
assert build_prefix(m3) == [[0, 0, 0, 0], [0, 1, 3, 6], [0, 5, 12, 21], [0, 12, 27, 45]]
assert NumMatrix(m3).sumRegion(1, 1, 2, 2) == 28 == oracle(m3, 1, 1, 2, 2)


# Bài 3
class NumMatrixTodoAns:
    def __init__(self, matrix):
        m, n = len(matrix), len(matrix[0])
        self.P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                self.P[i + 1][j + 1] = matrix[i][j] + self.P[i][j + 1] + self.P[i + 1][j] - self.P[i][j]

    def sumRegion(self, row1, col1, row2, col2):
        P = self.P
        return P[row2 + 1][col2 + 1] - P[row1][col2 + 1] - P[row2 + 1][col1] + P[row1][col1]


assert run(NumMatrixTodoAns, examples_ops, examples_args) == examples_expected


# Bài 4
class NumMatrix1DRows(NumMatrixRows):
    pass


t0 = time.perf_counter()
run(NumMatrix1DRows, ops, args)
t_rows = time.perf_counter() - t0
t0 = time.perf_counter()
run(NumMatrix, ops, args)
t_2d = time.perf_counter() - t0
print(f"Cách 2 / Cách 3 trên 10^4 truy vấn: {t_rows / t_2d:.1f} lần")


# Bài 5
class NumMatrix3D:
    def __init__(self, cube):
        a, b, c = len(cube), len(cube[0]), len(cube[0][0])
        self.P = [[[0] * (c + 1) for _ in range(b + 1)] for _ in range(a + 1)]
        P = self.P
        for i in range(a):
            for j in range(b):
                for k in range(c):
                    P[i + 1][j + 1][k + 1] = (cube[i][j][k]
                                              + P[i][j + 1][k + 1] + P[i + 1][j][k + 1] + P[i + 1][j + 1][k]
                                              - P[i][j][k + 1] - P[i][j + 1][k] - P[i + 1][j][k]
                                              + P[i][j][k])

    def sumRegion(self, x1, y1, z1, x2, y2, z2):
        P = self.P
        total = 0
        for xi, sx in ((x2 + 1, 1), (x1, -1)):
            for yi, sy in ((y2 + 1, 1), (y1, -1)):
                for zi, sz in ((z2 + 1, 1), (z1, -1)):
                    total += sx * sy * sz * P[xi][yi][zi]
        return total


rng = random.Random(21)
cube = [[[rng.randint(-5, 5) for _ in range(4)] for _ in range(5)] for _ in range(3)]
nm3 = NumMatrix3D(cube)
for _ in range(200):
    x1, x2 = sorted((rng.randrange(3), rng.randrange(3)))
    y1, y2 = sorted((rng.randrange(5), rng.randrange(5)))
    z1, z2 = sorted((rng.randrange(4), rng.randrange(4)))
    brute3 = sum(cube[i][j][k] for i in range(x1, x2 + 1) for j in range(y1, y2 + 1) for k in range(z1, z2 + 1))
    assert nm3.sumRegion(x1, y1, z1, x2, y2, z2) == brute3
print("PASS đáp án bài tập")

Cách 2 / Cách 3 trên 10^4 truy vấn: 7.4 lần
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng tiền tố 2D có đệm)

```python
class NumMatrix:
    def __init__(self, matrix: List[List[int]]):
        m, n = len(matrix), len(matrix[0])
        self.s = [[0] * (n + 1) for _ in range(m + 1)]
        for i, row in enumerate(matrix):
            for j, v in enumerate(row):
                self.s[i + 1][j + 1] = self.s[i][j + 1] + self.s[i + 1][j] - self.s[i][j] + v

    def sumRegion(self, row1: int, col1: int, row2: int, col2: int) -> int:
        return (self.s[row2 + 1][col2 + 1] - self.s[row2 + 1][col1]
                - self.s[row1][col2 + 1] + self.s[row1][col1])
```

**Checklist**

- [ ] `P` có kích thước `(m+1) × (n+1)` với hàng/cột đệm `0`?
- [ ] Dựng: `P[i+1][j+1] = P[i][j+1] + P[i+1][j] − P[i][j] + matrix[i][j]`?
- [ ] Truy vấn: `P[r2+1][c2+1] − P[r2+1][c1] − P[r1][c2+1] + P[r1][c1]` (khối lớn − dải − dải + góc)?
- [ ] Không dùng chỉ số `-1` (quay vòng) và không lệch `+1`?
- [ ] Tiền xử lý trong `__init__`, mỗi `sumRegion` `O(1)`?
- [ ] Đã thử ma trận `1×1`, một hàng/cột, cả bảng, ô đơn, giá trị âm?
- [ ] Đã dựng `run(cls, ops, args)` và so với brute force bằng lát cắt?
- [ ] Skeleton đúng `class NumMatrix` / `__init__(self, matrix)` / `sumRegion(self, row1, col1, row2, col2)`?